In [ ]:
import model
import epidemic_simulation
import survey_design
import random
import numpy as np
import scipy.stats
import matplotlib
import matplotlib.pyplot as plt
import matplotlib as mpl
import matplotlib.cm as cm
import arviz
import cmdstanpy
from cmdstanpy import CmdStanModel

In [ ]:
f_infectious, f_recovered, delay_inf, delay_recov = epidemic_simulation.delays()
transmission_rate = epidemic_simulation.transmission_rate
f_var = 100 ** 2

In [ ]:
inference_model = CmdStanModel(stan_file='model_rw2.stan', cpp_options={'STAN_THREADS':'true'})

spacings = [21, 84]
sample_sizes = [2000,]

m = model.AgentModel(1000000, 50)

full_spacings = np.repeat(spacings, len(sample_sizes))
full_sample_sizes = np.tile(sample_sizes, len(spacings))

surveys = survey_design.make_surveys(m, full_spacings, full_sample_sizes, full_spacings, full_sample_sizes, duration=7)
df = survey_design.run_simulations(m, surveys, transmission_rate, f_infectious, f_recovered)
fits = survey_design.run_inference(df, surveys, inference_model, delay_inf, delay_recov, 'misspec_true', init_mcmc=800)


f_mean_err0 = 100
theta = f_var / f_mean_err0
k = f_mean_err0 / theta
f_dist = scipy.stats.gamma(k, scale=theta)
f_recovered_err0 = f_dist.pdf(np.arange(2000))
num_samples = 100000
delay_recov_err0 = [0] * 10000
for _ in range(num_samples):
    num_days = model.sample_discrete(f_recovered_err0) + model.sample_discrete(f_infectious)
    for i in range(num_days):
        delay_recov_err0[i] += 1 
delay_recov_err0 = np.asarray(delay_recov_err0) / num_samples
delay_recov_err0 = delay_recov_err0[:365]

fits_err0 = survey_design.run_inference(df, surveys, inference_model, delay_inf, delay_recov_err0, 'misspec_err0', init_mcmc=800)



f_mean_err1 = 500
theta = f_var / f_mean_err1
k = f_mean_err1 / theta
f_dist = scipy.stats.gamma(k, scale=theta)
f_recovered_err1 = f_dist.pdf(np.arange(2000))
num_samples = 100000
delay_recov_err1 = [0] * 10000
for _ in range(num_samples):
    num_days = model.sample_discrete(f_recovered_err1) + model.sample_discrete(f_infectious)
    for i in range(num_days):
        delay_recov_err1[i] += 1 
delay_recov_err1 = np.asarray(delay_recov_err1) / num_samples
delay_recov_err1 = delay_recov_err1[:365]

fits_err1 = survey_design.run_inference(df, surveys, inference_model, delay_inf, delay_recov_err1, 'misspec_err1', init_mcmc=800)


In [ ]:
fig = plt.figure(figsize=(9, 6))

ax = fig.add_subplot(3, 3, 2)

delay_times = np.arange(len(delay_recov))
l1, = ax.plot(delay_times, delay_recov, color='k', ls='-', label='300 (Correct)')
l2, = ax.plot(delay_times, delay_recov_err0, color='k', ls=':', label='150')
l3, = ax.plot(delay_times, delay_recov_err1, color='k', ls='--', label='600')

ax.set_xlabel('Time (days)')
ax.set_ylabel('Probability')

ax.spines[['right', 'top']].set_visible(False)


ax = fig.add_subplot(3, 3, 3)
ax.spines[['right', 'top', 'left', 'bottom']].set_visible(False)
ax.set_xticks([])
ax.set_yticks([])

ax.legend((l1, l2, l3,), ['300 (Correct)', '150', '600'], loc='center left', title='Mean')



ax = fig.add_subplot(3, 3, 4)

ax.set_title('Mean=150')

fit = fits_err0[0]
idx1 = 0

for prev_surv_start in surveys[2 * idx1].start_dates:
    if prev_surv_start not in surveys[2 * idx1 + 1].start_dates:
        l1 = ax.axvspan(prev_surv_start - 3.5, prev_surv_start + 3.5, alpha=0.25, color='yellowgreen')
    else:
        l2 = ax.axvspan(prev_surv_start - 3.5, prev_surv_start + 3.5, alpha=0.25, color='gray')
for sero_surv_start in surveys[2 * idx1 + 1].start_dates:
    if sero_surv_start not in surveys[2 * idx1].start_dates:
        l3 = ax.axvspan(sero_surv_start - 3.5, sero_surv_start + 3.5, alpha=0.25, color='tab:orange')
    else:
        pass

ax.plot(df['transmissions'], color='k', label='Incidence')

x = np.asarray(fit.incidence[:, :])
lower = np.percentile(x, 5, axis=0,)
upper = np.percentile(x, 95, axis=0,)
mid = np.median(x, axis=0)
tplot = np.arange(len(mid))

ax.plot(tplot, mid,)
ax.fill_between(tplot, lower, upper, alpha=0.35)
ax.spines[['right', 'top']].set_visible(False)

ax.set_ylabel('Incidence')

ax.tick_params(labelbottom=False, labelleft=True)




ax = fig.add_subplot(3, 3, 5)

ax.set_title('Mean=300 (Correct)')

fit = fits[0]
idx1 = 0

for prev_surv_start in surveys[2 * idx1].start_dates:
    if prev_surv_start not in surveys[2 * idx1 + 1].start_dates:
        l1 = ax.axvspan(prev_surv_start - 3.5, prev_surv_start + 3.5, alpha=0.25, color='yellowgreen')
    else:
        l2 = ax.axvspan(prev_surv_start - 3.5, prev_surv_start + 3.5, alpha=0.25, color='gray')
for sero_surv_start in surveys[2 * idx1 + 1].start_dates:
    if sero_surv_start not in surveys[2 * idx1].start_dates:
        l3 = ax.axvspan(sero_surv_start - 3.5, sero_surv_start + 3.5, alpha=0.25, color='tab:orange')
    else:
        pass

ax.plot(df['transmissions'], color='k', label='Incidence')

x = np.asarray(fit.incidence[:, :])
lower = np.percentile(x, 5, axis=0,)
upper = np.percentile(x, 95, axis=0,)
mid = np.median(x, axis=0)
tplot = np.arange(len(mid))

ax.plot(tplot, mid,)
ax.fill_between(tplot, lower, upper, alpha=0.35)
ax.spines[['right', 'top']].set_visible(False)


ax.tick_params(labelbottom=False, labelleft=False)





ax = fig.add_subplot(3, 3, 6)

ax.set_title('Mean=600')

fit = fits_err1[0]
idx1 = 0

for prev_surv_start in surveys[2 * idx1].start_dates:
    if prev_surv_start not in surveys[2 * idx1 + 1].start_dates:
        l1 = ax.axvspan(prev_surv_start - 3.5, prev_surv_start + 3.5, alpha=0.25, color='yellowgreen')
    else:
        l2 = ax.axvspan(prev_surv_start - 3.5, prev_surv_start + 3.5, alpha=0.25, color='gray')
for sero_surv_start in surveys[2 * idx1 + 1].start_dates:
    if sero_surv_start not in surveys[2 * idx1].start_dates:
        l3 = ax.axvspan(sero_surv_start - 3.5, sero_surv_start + 3.5, alpha=0.25, color='tab:orange')
    else:
        pass

ax.plot(df['transmissions'], color='k', label='Incidence')

x = np.asarray(fit.incidence[:, :])
lower = np.percentile(x, 5, axis=0,)
upper = np.percentile(x, 95, axis=0,)
mid = np.median(x, axis=0)
tplot = np.arange(len(mid))

ax.plot(tplot, mid,)
ax.fill_between(tplot, lower, upper, alpha=0.35)
ax.spines[['right', 'top']].set_visible(False)


ax.tick_params(labelbottom=False, labelleft=False)










ax = fig.add_subplot(3, 3, 7)

fit = fits_err0[1]
idx1 = 1

for prev_surv_start in surveys[2 * idx1].start_dates:
    if prev_surv_start not in surveys[2 * idx1 + 1].start_dates:
        l1 = ax.axvspan(prev_surv_start - 3.5, prev_surv_start + 3.5, alpha=0.25, color='yellowgreen')
    else:
        l2 = ax.axvspan(prev_surv_start - 3.5, prev_surv_start + 3.5, alpha=0.25, color='gray')
for sero_surv_start in surveys[2 * idx1 + 1].start_dates:
    if sero_surv_start not in surveys[2 * idx1].start_dates:
        l3 = ax.axvspan(sero_surv_start - 3.5, sero_surv_start + 3.5, alpha=0.25, color='tab:orange')
    else:
        pass

ax.plot(df['transmissions'], color='k', label='Incidence')

x = np.asarray(fit.incidence[:, :])
lower = np.percentile(x, 5, axis=0,)
upper = np.percentile(x, 95, axis=0,)
mid = np.median(x, axis=0)
tplot = np.arange(len(mid))

ax.plot(tplot, mid,)
ax.fill_between(tplot, lower, upper, alpha=0.35)
ax.spines[['right', 'top']].set_visible(False)

ax.set_ylabel('Incidence')
ax.set_xlabel('Time (days)')
ax.tick_params(labelbottom=True, labelleft=True)




ax = fig.add_subplot(3, 3, 8)

fit = fits[1]
idx1 = 1

for prev_surv_start in surveys[2 * idx1].start_dates:
    if prev_surv_start not in surveys[2 * idx1 + 1].start_dates:
        l1 = ax.axvspan(prev_surv_start - 3.5, prev_surv_start + 3.5, alpha=0.25, color='yellowgreen')
    else:
        l2 = ax.axvspan(prev_surv_start - 3.5, prev_surv_start + 3.5, alpha=0.25, color='gray')
for sero_surv_start in surveys[2 * idx1 + 1].start_dates:
    if sero_surv_start not in surveys[2 * idx1].start_dates:
        l3 = ax.axvspan(sero_surv_start - 3.5, sero_surv_start + 3.5, alpha=0.25, color='tab:orange')
    else:
        pass

ax.plot(df['transmissions'], color='k', label='Incidence')

x = np.asarray(fit.incidence[:, :])
lower = np.percentile(x, 5, axis=0,)
upper = np.percentile(x, 95, axis=0,)
mid = np.median(x, axis=0)
tplot = np.arange(len(mid))

ax.plot(tplot, mid,)
ax.fill_between(tplot, lower, upper, alpha=0.35)
ax.spines[['right', 'top']].set_visible(False)

ax.set_xlabel('Time (days)')

ax.tick_params(labelbottom=True, labelleft=False)




ax = fig.add_subplot(3, 3, 9)

fit = fits_err1[1]
idx1 = 1

for prev_surv_start in surveys[2 * idx1].start_dates:
    if prev_surv_start not in surveys[2 * idx1 + 1].start_dates:
        l1 = ax.axvspan(prev_surv_start - 3.5, prev_surv_start + 3.5, alpha=0.25, color='yellowgreen')
    else:
        l2 = ax.axvspan(prev_surv_start - 3.5, prev_surv_start + 3.5, alpha=0.25, color='gray')
for sero_surv_start in surveys[2 * idx1 + 1].start_dates:
    if sero_surv_start not in surveys[2 * idx1].start_dates:
        l3 = ax.axvspan(sero_surv_start - 3.5, sero_surv_start + 3.5, alpha=0.25, color='tab:orange')
    else:
        pass

ax.plot(df['transmissions'], color='k', label='Incidence')

x = np.asarray(fit.incidence[:, :])
lower = np.percentile(x, 5, axis=0,)
upper = np.percentile(x, 95, axis=0,)
mid = np.median(x, axis=0)
tplot = np.arange(len(mid))

ax.plot(tplot, mid,)
ax.fill_between(tplot, lower, upper, alpha=0.35)
ax.spines[['right', 'top']].set_visible(False)

ax.set_xlabel('Time (days)')

ax.tick_params(labelbottom=True, labelleft=False)



fig.set_tight_layout(True)

plt.savefig('Figure7.pdf')
plt.show()

